## v4.1 — maximum-effort legitimate push (final submission candidate)

Competition ends today with limited submissions left, so this notebook is
built to squeeze out every remaining honest gain **and** to make the final
decision rule as robust as possible, since we can't afford exploratory
submissions to test it.

### The key insight driving every choice here

F1-optimal thresholding is **invariant to any monotone transform of the
scores**. That means probability calibration (Platt/isotonic) is pointless
here -- it can't change which rows end up above the cutoff. The *only* thing
that raises the achievable F1 ceiling is **better ranking quality** (AUC).
So this notebook spends its entire budget on things that improve ranking:

1. **More model diversity** -- 6 algorithms (XGBoost, LightGBM, CatBoost,
   HistGB, RandomForest, ExtraTrees) rather than 4. Weak-but-different models
   can still help a weighted blend; the weight search drops them to ~0 if not.
2. **Bigger hyperparameter search** -- 40/40/30/25/15/15 random-search
   iterations instead of v3.1's 12.
3. **Seed averaging** -- each tuned model is run under 3 seeds and averaged,
   which decorrelates fitting noise and improves ranking stability.
4. **10-fold instead of 5-fold** for final evaluation -- more training data
   per fold, and a more reliable threshold estimate (the thing we can't
   validate by submitting).
5. **Stacking vs. weighted blending**, whichever actually wins out-of-fold.

Everything targets `pos_label=0`, the actual graded convention (see
`docs/Nishkarsh/scoring_convention.md`). `customer_id`/`last_name` remain
excluded as leakage.

### On the decision rule (this matters more than usual here)

Test predictions are averaged across all fold-models and seeds, which makes
their score distribution *tighter* than the OOF distribution (an average of
30 models vs. an average of 3). A raw probability threshold tuned on OOF
doesn't necessarily transfer to a differently-shaped distribution. So this
notebook also computes a **rate-based rule** -- label the top q% of test rows
as churn, where q is the churn rate the optimal OOF rule produced -- which is
invariant to that shift. Both are reported, along with how much they actually
disagree.


In [13]:
import gc, time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
POS_LABEL = 1          # confirmed with organizers: F1 graded with non-churn as positive
SEEDS = [42, 202, 777]
rng = np.random.RandomState(RANDOM_STATE)
t_start = time.time()

def elapsed():
    return f"[{time.time() - t_start:6.1f}s]"


In [14]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')

ID_COLS = ['id', 'customer_id', 'last_name']
TARGET = 'exit_status'
PROD_COUNT_IMPUTE_FEATURES = [
    'age', 'is_active', 'acc_balance', 'country', 'credit_score',
    'has_card', 'estimated_salary', 'tenure',
]
BASE_NUM_COLS = ['credit_score', 'age', 'tenure', 'acc_balance', 'has_card', 'is_active', 'estimated_salary']
CAT_COLS = ['country', 'gender', 'prod_count']


def make_X(df):
    return df.drop(columns=[c for c in ID_COLS + [TARGET] if c in df.columns])


X = make_X(train)
y = train[TARGET]
X_test_raw = make_X(test)
print("train:", X.shape, "test:", X_test_raw.shape)


train: (90000, 10) test: (30000, 10)


In [15]:
class BankChurnImputer(BaseEstimator, TransformerMixin):
    def __init__(self, prod_count_features=PROD_COUNT_IMPUTE_FEATURES, random_state=RANDOM_STATE):
        self.prod_count_features = prod_count_features
        self.random_state = random_state

    def fit(self, X, y=None):
        X = X.copy()
        X['country'] = X['country'].fillna('Unknown')
        self.balance_median_by_country_ = X.groupby('country')['acc_balance'].median()
        self.balance_global_median_ = X['acc_balance'].median()
        self.credit_score_median_ = X['credit_score'].median()

        X['acc_balance'] = X['acc_balance'].fillna(X['country'].map(self.balance_median_by_country_))
        X['acc_balance'] = X['acc_balance'].fillna(self.balance_global_median_)
        X['credit_score'] = X['credit_score'].fillna(self.credit_score_median_)

        known = X.dropna(subset=['prod_count'])
        Xk = pd.get_dummies(known[self.prod_count_features], columns=['country'])
        self.prod_count_columns_ = Xk.columns
        self.prod_count_model_ = RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, random_state=self.random_state, n_jobs=-1)
        self.prod_count_model_.fit(Xk, known['prod_count'].astype(int))
        return self

    def transform(self, X):
        X = X.copy()
        X['country'] = X['country'].fillna('Unknown')
        X['acc_balance'] = X['acc_balance'].fillna(X['country'].map(self.balance_median_by_country_))
        X['acc_balance'] = X['acc_balance'].fillna(self.balance_global_median_)
        X['credit_score'] = X['credit_score'].fillna(self.credit_score_median_)

        missing = X['prod_count'].isna()
        if missing.any():
            Xm = pd.get_dummies(X.loc[missing, self.prod_count_features], columns=['country'])
            Xm = Xm.reindex(columns=self.prod_count_columns_, fill_value=0)
            X.loc[missing, 'prod_count'] = self.prod_count_model_.predict(Xm)
        return X


def add_engineered_features(df):
    """v2's feature set. Whether it helps under pos_label=0 is tested below,
    not assumed -- under pos_label=1 it was slightly negative."""
    df = df.copy()
    df['balance_zero'] = (df['acc_balance'] == 0).astype(float)
    df['balance_salary_ratio'] = df['acc_balance'] / df['estimated_salary'].replace(0, np.nan)
    df['balance_age_ratio'] = df['acc_balance'] / df['age'].replace(0, np.nan)
    df['active_x_prod'] = df['is_active'] * df['prod_count']
    df['age_x_active'] = df['age'] * df['is_active']
    for c in ['balance_salary_ratio', 'balance_age_ratio']:
        df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df


FE_COLS = ['balance_zero', 'balance_salary_ratio', 'balance_age_ratio', 'active_x_prod', 'age_x_active']


### Cached preprocessed folds

Imputation + target encoding are model-independent, so they run once per fold
layout and every search/evaluation reuses the cached arrays. `use_fe` toggles
the v2 engineered features so we can measure whether they help under the
corrected metric rather than assuming.


In [16]:
def build_folds(X, y, n_splits, use_fe, random_state=RANDOM_STATE, X_test=None):
    num_cols = BASE_NUM_COLS + (FE_COLS if use_fe else [])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    folds = []
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr = y.iloc[tr_idx]

        imp = BankChurnImputer(); imp.fit(X_tr)
        X_tr, X_va = imp.transform(X_tr), imp.transform(X_va)
        if use_fe:
            X_tr, X_va = add_engineered_features(X_tr), add_engineered_features(X_va)

        te = TargetEncoder(target_type='binary', random_state=random_state, cv=5)
        tr_cat = te.fit_transform(X_tr[CAT_COLS], y_tr)
        va_cat = te.transform(X_va[CAT_COLS])

        entry = {
            'X_tr': np.hstack([X_tr[num_cols].to_numpy(), tr_cat]),
            'y_tr': y_tr.to_numpy(),
            'X_va': np.hstack([X_va[num_cols].to_numpy(), va_cat]),
            'va_idx': va_idx,
        }
        if X_test is not None:
            X_te = imp.transform(X_test.copy())
            if use_fe:
                X_te = add_engineered_features(X_te)
            entry['X_te'] = np.hstack([X_te[num_cols].to_numpy(), te.transform(X_te[CAT_COLS])])
        folds.append(entry)
    return folds


y_arr = y.to_numpy()

def f1_at(scores, thr):
    return f1_score(y_arr, scores > thr, pos_label=POS_LABEL)

def best_threshold(scores, y_true=None, n_grid=1500):
    """Fine threshold sweep over the score range."""
    y_true = y_arr if y_true is None else y_true
    lo, hi = np.quantile(scores, 0.01), np.quantile(scores, 0.99)
    grid = np.linspace(lo, hi, n_grid)
    f1s = np.array([f1_score(y_true, scores > t, pos_label=POS_LABEL) for t in grid])
    i = int(np.argmax(f1s))
    return grid[i], f1s[i], grid, f1s


print(f"{elapsed()} building 3-fold search cache (no FE)...")
search_folds_nofe = build_folds(X, y, 3, use_fe=False)
print(f"{elapsed()} building 3-fold search cache (with FE)...")
search_folds_fe = build_folds(X, y, 3, use_fe=True)
print(f"{elapsed()} done.")


[   0.2s] building 3-fold search cache (no FE)...
[   8.3s] building 3-fold search cache (with FE)...
[  16.3s] done.


In [17]:
def evaluate(build_fn, folds, seeds=(RANDOM_STATE,), collect_test=False):
    """OOF probabilities averaged over seeds; optionally bagged test predictions."""
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test_raw)) if collect_test else None
    n_models = 0
    for seed in seeds:
        for f in folds:
            m = build_fn(seed)
            sw = compute_sample_weight('balanced', f['y_tr'])
            m.fit(f['X_tr'], f['y_tr'], sample_weight=sw)
            oof[f['va_idx']] += m.predict_proba(f['X_va'])[:, 1]
            if collect_test:
                test_pred += m.predict_proba(f['X_te'])[:, 1]
                n_models += 1
            del m
        gc.collect()
    oof /= len(seeds)
    if collect_test:
        test_pred /= n_models
    return oof, test_pred


def random_search(factory, space, folds, n_iter, tag, random_state=RANDOM_STATE):
    local_rng = np.random.RandomState(random_state)
    rows, best = [], {'f1': -1.0}
    for i in range(n_iter):
        params = {k: v[local_rng.randint(len(v))] for k, v in space.items()}
        oof, _ = evaluate(lambda s, p=params: factory(seed=s, **p), folds)
        thr, f1, _, _ = best_threshold(oof)
        rows.append({**params, 'f1': f1})
        if f1 > best['f1']:
            best = {'f1': f1, 'params': params}
    print(f"{elapsed()} {tag}: best 3-fold F1={best['f1']:.5f} params={best['params']}")
    return best, pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)


### Pre-check: do the v2 engineered features help under the corrected metric?

Cheap to answer (two XGBoost evaluations), and it decides the feature set for
everything downstream, so it's worth answering rather than assuming.


In [18]:
probe = dict(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
             colsample_bytree=0.9, min_child_weight=5, reg_lambda=1.0)

def xgb_factory(seed=RANDOM_STATE, **p):
    return XGBClassifier(**p, tree_method='hist', random_state=seed, verbosity=0, n_jobs=-1)

oof_nofe, _ = evaluate(lambda s: xgb_factory(seed=s, **probe), search_folds_nofe)
oof_fe, _ = evaluate(lambda s: xgb_factory(seed=s, **probe), search_folds_fe)
_, f1_nofe, _, _ = best_threshold(oof_nofe)
_, f1_fe, _, _ = best_threshold(oof_fe)
auc_nofe, auc_fe = roc_auc_score(y_arr, oof_nofe), roc_auc_score(y_arr, oof_fe)

USE_FE = f1_fe > f1_nofe
print(f"{elapsed()} without FE: F1={f1_nofe:.5f} AUC={auc_nofe:.5f}")
print(f"{elapsed()} with FE:    F1={f1_fe:.5f} AUC={auc_fe:.5f}")
print(f"--> USE_FE = {USE_FE}")

search_folds = search_folds_fe if USE_FE else search_folds_nofe
if USE_FE:
    search_folds_nofe = None
else:
    search_folds_fe = None
gc.collect()


[  40.4s] without FE: F1=0.65290 AUC=0.87932
[  40.4s] with FE:    F1=0.65372 AUC=0.87917
--> USE_FE = True


0

### Hyperparameter search (3-fold), one model at a time

Sequential by construction -- only CatBoost uses the GPU, and nothing runs
concurrently, so the 6 GB ceiling is never contended.


In [19]:
def lgb_factory(seed=RANDOM_STATE, **p):
    return LGBMClassifier(**p, random_state=seed, verbose=-1, n_jobs=-1)

def cat_factory(seed=RANDOM_STATE, **p):
    return CatBoostClassifier(**p, task_type='GPU', devices='0', random_seed=seed, verbose=False)

def hgb_factory(seed=RANDOM_STATE, **p):
    return HistGradientBoostingClassifier(**p, random_state=seed)

def rf_factory(seed=RANDOM_STATE, **p):
    return RandomForestClassifier(**p, random_state=seed, n_jobs=-1)

def et_factory(seed=RANDOM_STATE, **p):
    return ExtraTreesClassifier(**p, random_state=seed, n_jobs=-1)

SPACES = {
    'xgb': (xgb_factory, 40, {
        'n_estimators': [300, 400, 600, 800, 1000],
        'max_depth': [3, 4, 5, 6, 7, 8],
        'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        'min_child_weight': [1, 3, 5, 7, 10, 15],
        'reg_lambda': [0.5, 1.0, 2.0, 5.0, 10.0],
        'reg_alpha': [0.0, 0.1, 0.5, 1.0],
    }),
    'lgb': (lgb_factory, 40, {
        'n_estimators': [300, 400, 600, 800, 1000],
        'num_leaves': [15, 31, 63, 127, 255],
        'max_depth': [-1, 4, 6, 8, 10],
        'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'subsample_freq': [1],
        'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        'min_child_samples': [5, 10, 20, 30, 50],
        'reg_lambda': [0.0, 0.5, 1.0, 2.0, 5.0],
    }),
    'cat': (cat_factory, 30, {
        'iterations': [300, 400, 600, 800],
        'depth': [4, 5, 6, 7, 8],
        'learning_rate': [0.02, 0.03, 0.05, 0.08],
        'l2_leaf_reg': [1.0, 3.0, 5.0, 7.0, 9.0],
    }),
    'hgb': (hgb_factory, 25, {
        'max_iter': [300, 400, 600, 800],
        'max_depth': [4, 5, 6, 7, None],
        'learning_rate': [0.02, 0.03, 0.05, 0.08],
        'l2_regularization': [0.0, 0.5, 1.0, 2.0],
        'max_leaf_nodes': [15, 31, 63, 127],
        'min_samples_leaf': [10, 20, 40],
    }),
    'rf': (rf_factory, 15, {
        'n_estimators': [400, 600],
        'max_depth': [None, 12, 16, 20],
        'min_samples_leaf': [1, 2, 3, 5, 10],
        'max_features': ['sqrt', 0.4, 0.6],
    }),
    'et': (et_factory, 15, {
        'n_estimators': [400, 600],
        'max_depth': [None, 12, 16, 20],
        'min_samples_leaf': [1, 2, 3, 5, 10],
        'max_features': ['sqrt', 0.4, 0.6],
    }),
}

best_params = {}
for name, (factory, n_iter, space) in SPACES.items():
    best, _ = random_search(factory, space, search_folds, n_iter, name)
    best_params[name] = best['params']
    gc.collect()


[ 551.5s] xgb: best 3-fold F1=0.65651 params={'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.02, 'subsample': 0.9, 'colsample_bytree': 1.0, 'min_child_weight': 15, 'reg_lambda': 5.0, 'reg_alpha': 0.0}


c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fit

[1098.7s] lgb: best 3-fold F1=0.65604 params={'n_estimators': 300, 'num_leaves': 31, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 1.0, 'subsample_freq': 1, 'colsample_bytree': 0.7, 'min_child_samples': 5, 'reg_lambda': 2.0}
[1650.9s] cat: best 3-fold F1=0.65555 params={'iterations': 600, 'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 9.0}
[1970.6s] hgb: best 3-fold F1=0.65509 params={'max_iter': 800, 'max_depth': 4, 'learning_rate': 0.08, 'l2_regularization': 2.0, 'max_leaf_nodes': 127, 'min_samples_leaf': 40}
[2362.8s] rf: best 3-fold F1=0.65182 params={'n_estimators': 400, 'max_depth': 20, 'min_samples_leaf': 10, 'max_features': 'sqrt'}
[2638.6s] et: best 3-fold F1=0.65096 params={'n_estimators': 400, 'max_depth': 20, 'min_samples_leaf': 10, 'max_features': 0.6}


### Final 10-fold evaluation, 3 seeds per model, bagged test predictions

10 folds gives each model more training data and makes the threshold estimate
more reliable -- which matters here because the threshold is the one decision
we cannot validate without spending a submission. Test predictions are
accumulated from every fold-model and seed (30 models per algorithm), which
is free bagging on top.


In [20]:
print(f"{elapsed()} building 10-fold final cache (with test transforms)...")
final_folds = build_folds(X, y, 10, use_fe=USE_FE, X_test=X_test_raw)
print(f"{elapsed()} done.")

FACTORIES = {k: v[0] for k, v in SPACES.items()}
model_names = list(SPACES.keys())
oof_cols, test_cols = {}, {}

for name in model_names:
    factory, params = FACTORIES[name], best_params[name]
    oof_m, test_m = evaluate(lambda s, p=params, f=factory: f(seed=s, **p),
                             final_folds, seeds=SEEDS, collect_test=True)
    oof_cols[name], test_cols[name] = oof_m, test_m
    thr, f1, _, _ = best_threshold(oof_m)
    print(f"{elapsed()} {name}: 10-fold OOF F1={f1:.5f} AUC={roc_auc_score(y_arr, oof_m):.5f} thr={thr:.3f}")
    gc.collect()

oof_matrix = np.column_stack([oof_cols[n] for n in model_names])
test_matrix = np.column_stack([test_cols[n] for n in model_names])


[2638.6s] building 10-fold final cache (with test transforms)...
[2670.8s] done.
[2715.9s] xgb: 10-fold OOF F1=0.65747 AUC=0.88112 thr=0.631


c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fit

[2739.7s] lgb: 10-fold OOF F1=0.65706 AUC=0.88112 thr=0.640
[2813.6s] cat: 10-fold OOF F1=0.65682 AUC=0.88120 thr=0.623
[2841.9s] hgb: 10-fold OOF F1=0.65634 AUC=0.88089 thr=0.638
[2982.5s] rf: 10-fold OOF F1=0.65193 AUC=0.87766 thr=0.549
[3166.7s] et: 10-fold OOF F1=0.65043 AUC=0.87739 thr=0.609


### Combining: equal blend vs. weighted blend vs. stacking

The weighted blend's weights are fit on the same OOF vector we score on, so
it can flatter itself. To check that, weights are also fit on one half of the
OOF and scored on the other half -- if the held-out half agrees, the gain is
real rather than fitted noise.


In [21]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

# 1. equal-weight blend
equal_blend = oof_matrix.mean(axis=1)
thr_eq, f1_eq, _, _ = best_threshold(equal_blend)
print(f"equal blend      : F1={f1_eq:.5f} AUC={roc_auc_score(y_arr, equal_blend):.5f}")

# 2. weighted blend (dirichlet search)
best_w, best_wf1 = np.ones(len(model_names)) / len(model_names), f1_eq
for _ in range(2000):
    w = rng.dirichlet(np.ones(len(model_names)) * 0.8)
    _, f1, _, _ = best_threshold(oof_matrix @ w, n_grid=400)
    if f1 > best_wf1:
        best_wf1, best_w = f1, w
thr_w, f1_w, _, _ = best_threshold(oof_matrix @ best_w)
print(f"weighted blend   : F1={f1_w:.5f} weights={dict(zip(model_names, best_w.round(3)))}")

# honest check: fit weights on half the OOF, score on the other half
half = rng.permutation(len(X))
h1, h2 = half[:len(X)//2], half[len(X)//2:]
w_h1, f1_h1 = np.ones(len(model_names)) / len(model_names), -1
for _ in range(1000):
    w = rng.dirichlet(np.ones(len(model_names)) * 0.8)
    s = oof_matrix[h1] @ w
    grid = np.linspace(np.quantile(s, .01), np.quantile(s, .99), 300)
    f1s = [f1_score(y_arr[h1], s > t, pos_label=POS_LABEL) for t in grid]
    if max(f1s) > f1_h1:
        f1_h1, w_h1, thr_h1 = max(f1s), w, grid[int(np.argmax(f1s))]
f1_h2_weighted = f1_score(y_arr[h2], (oof_matrix[h2] @ w_h1) > thr_h1, pos_label=POS_LABEL)
s_eq_h2 = equal_blend[h2]
grid = np.linspace(np.quantile(equal_blend[h1], .01), np.quantile(equal_blend[h1], .99), 300)
f1s_h1_eq = [f1_score(y_arr[h1], equal_blend[h1] > t, pos_label=POS_LABEL) for t in grid]
thr_eq_h1 = grid[int(np.argmax(f1s_h1_eq))]
f1_h2_equal = f1_score(y_arr[h2], s_eq_h2 > thr_eq_h1, pos_label=POS_LABEL)
print(f"holdout check    : weighted={f1_h2_weighted:.5f} vs equal={f1_h2_equal:.5f} "
      f"({'weighted holds up' if f1_h2_weighted >= f1_h2_equal else 'weighted was overfitting -> prefer equal'})")

# 3. logistic-regression stacking on logit-transformed OOF probabilities
stack_X = logit(oof_matrix)
stack_oof = cross_val_predict(
    LogisticRegression(max_iter=2000, C=1.0), stack_X, y_arr,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    method='predict_proba')[:, 1]
thr_st, f1_st, _, _ = best_threshold(stack_oof)
print(f"LR stacking      : F1={f1_st:.5f} AUC={roc_auc_score(y_arr, stack_oof):.5f}")


equal blend      : F1=0.65698 AUC=0.88120
weighted blend   : F1=0.65830 weights={'xgb': np.float64(0.19), 'lgb': np.float64(0.292), 'cat': np.float64(0.384), 'hgb': np.float64(0.122), 'rf': np.float64(0.011), 'et': np.float64(0.002)}
holdout check    : weighted=0.65359 vs equal=0.65404 (weighted was overfitting -> prefer equal)
LR stacking      : F1=0.65768 AUC=0.88135


In [22]:
candidates = {
    'equal_blend': (equal_blend, test_matrix.mean(axis=1), f1_eq),
    'weighted_blend': (oof_matrix @ best_w, test_matrix @ best_w, f1_w),
}
# only trust the weighted blend if it survived the holdout check
if f1_h2_weighted < f1_h2_equal:
    candidates.pop('weighted_blend')

stacker = LogisticRegression(max_iter=2000, C=1.0).fit(logit(oof_matrix), y_arr)
candidates['lr_stack'] = (stack_oof, stacker.predict_proba(logit(test_matrix))[:, 1], f1_st)

best_name = max(candidates, key=lambda k: candidates[k][2])
oof_best, test_best, f1_best = candidates[best_name]
print(f"\nselected: {best_name}  OOF F1={f1_best:.5f}")

summary = pd.DataFrame(
    [{'model': n, 'oof_f1': best_threshold(oof_cols[n])[1], 'auc': roc_auc_score(y_arr, oof_cols[n])}
     for n in model_names] +
    [{'model': f'** {k} **', 'oof_f1': v[2], 'auc': roc_auc_score(y_arr, v[0])} for k, v in candidates.items()]
).sort_values('oof_f1', ascending=False).reset_index(drop=True)
summary



selected: lr_stack  OOF F1=0.65768


,model,oof_f1,auc
0,** lr_stack **,0.657683,0.881354
1,xgb,0.657467,0.881119
2,lgb,0.657062,0.881117
3,** equal_blend **,0.656977,0.881204
4,cat,0.656823,0.881201
5,hgb,0.656336,0.880893
6,rf,0.651934,0.877658
7,et,0.650431,0.877386


### Decision rule and submission

Two rules compared: the raw OOF probability threshold, and a rate-based rule
(label the top q% by score as churn, q taken from the OOF-optimal rule). The
rate-based rule is immune to the distribution shift caused by test
predictions being an average of 30 models while OOF is an average of 3.


In [23]:
import os

thr_best, f1_final, grid, f1_curve = best_threshold(oof_best)
oof_pred = (oof_best > thr_best).astype(int)
q = 1.0 - oof_pred.mean()          # fraction predicted NON-churn on OOF

# plateau check: how flat is the F1 curve near the optimum?
near = grid[f1_curve >= f1_final - 0.0005]
print(f"optimal threshold {thr_best:.4f} (plateau within 0.0005: {near.min():.4f}-{near.max():.4f})")
print(f"OOF F1={f1_final:.5f}, OOF churn rate={oof_pred.mean():.4f}")

# rule A: probability threshold
pred_thr = (test_best > thr_best).astype(int)
# rule B: rate-matched
cutoff = np.quantile(test_best, q)
pred_rate = (test_best > cutoff).astype(int)

disagree = (pred_thr != pred_rate).mean()
print(f"test churn rate -- threshold rule: {pred_thr.mean():.4f}, rate rule: {pred_rate.mean():.4f}")
print(f"the two rules disagree on {disagree*100:.2f}% of test rows")

os.makedirs('outputs', exist_ok=True)
pd.DataFrame({'id': test['id'], 'exit_status': pred_rate}).to_csv(
    'outputs/v4_1_primary_submission.csv', index=False)
pd.DataFrame({'id': test['id'], 'exit_status': pred_thr}).to_csv(
    'outputs/v4_1_alternate_submission.csv', index=False)

print(f"\n{elapsed()} wrote outputs/v4_1_primary_submission.csv (rate-matched rule)")
print(f"{elapsed()} wrote outputs/v4_1_alternate_submission.csv (probability-threshold rule)")
print(f"expected F1 ~= {f1_final:.4f}")


optimal threshold 0.3110 (plateau within 0.0005: 0.3067-0.3229)
OOF F1=0.65768, OOF churn rate=0.2403
test churn rate -- threshold rule: 0.2431, rate rule: 0.2403
the two rules disagree on 0.29% of test rows

[10530.1s] wrote outputs/v4_1_primary_submission.csv (rate-matched rule)
[10530.1s] wrote outputs/v4_1_alternate_submission.csv (probability-threshold rule)
expected F1 ~= 0.6577
